# PODCAST exploration, target construction, and SAILER-style splits

This notebook prepares the Hugging Face `AbstractTTS/PODCAST` data for the project structure:

```text
data/raw/podcast/...
data/manifests/podcast/...
data/processed/podcast/...
outputs/podcast_dataset_exploration/...
```

Important correction: the emotion columns are not separate files and they are not hard labels. A row can have a value for every emotion category because the row target is a **soft emotion distribution**. That is compatible with the paper's training logic: primary emotion prediction should use a distribution target, not only a one-hot label.

The notebook creates:

1. a lean manifest with row locators into the original Parquet shards;
2. native 16-emotion soft targets from the dataset columns;
3. paper-compatible 9-emotion primary soft targets;
4. majority/minority class metadata;
5. optional secondary/multilabel targets and emotion-attribute regression targets;
6. train/dev/test split manifests.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)


## 1. Project paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent # Notebook has root from /notebooks always
assert PROJECT_ROOT.match("sailer_ser_reproduction")

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "podcast"
MANIFEST_ROOT = PROJECT_ROOT / "data" / "manifests" / "podcast"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "podcast"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "podcast_dataset_exploration"

for path in [MANIFEST_ROOT, MANIFEST_ROOT / "splits", PROCESSED_ROOT, OUTPUT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dataset root:", RAW_ROOT)
print("Manifest root:", MANIFEST_ROOT)
print("Processed feature root:", PROCESSED_ROOT)
print("Exploration output root:", OUTPUT_ROOT)

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"Expected dataset under {RAW_ROOT}.\n"
        "Download it first with: python scripts/download_podcast_dataset.py"
    )


## 2. Paper target logic

The paper uses these primary emotions:

```text
A C D F H N S U O
angry, contempt, disgust, fear, happy, neutral, sad, surprise, other
```

The paper treats `neutral`, `happy`, `sad`, and `angry` as majority classes. The remaining primary classes are minority classes. The notebook keeps that exact primary target space.

The Hugging Face dataset exposes a richer 16-emotion set. We keep that native set for optional secondary/multilabel supervision, then project it into the 9-class primary target used by the paper.

The mapping below is explicit and editable. Categories that do not map cleanly into the paper's eight named emotions are assigned to `other` rather than forced into a questionable class.


In [ ]:
PAPER_EMOTIONS = ["angry", "contempt", "disgust", "fear", "happy", "neutral", "sad", "surprise", "other"]
MAJORITY_EMOTIONS = ["neutral", "happy", "sad", "angry"]
MINORITY_EMOTIONS = ["contempt", "disgust", "fear", "surprise", "other"]

NATIVE_EMOTION_COLS = [
    "angry", "frustrated", "annoyed", "disappointed", "sad", "disgust", "depressed", "contempt",
    "confused", "concerned", "fear", "neutral", "surprise", "amused", "excited", "happy",
]

PAPER_MAPPING = {
    "angry": ["angry", "frustrated", "annoyed"],
    "contempt": ["contempt"],
    "disgust": ["disgust"],
    "fear": ["fear"],
    "happy": ["happy", "amused", "excited"],
    "neutral": ["neutral"],
    "sad": ["sad", "disappointed", "depressed"],
    "surprise": ["surprise"],
    "other": ["confused", "concerned"],
}

ATTRIBUTE_COLS = ["EmoAct", "EmoVal", "EmoDom"]
TEXT_CANDIDATES = ["transcription", "transcript", "text", "sentence", "caption"]
FILE_CANDIDATES = ["file", "filename", "path", "audio_id", "id"]
AUDIO_CANDIDATES = ["audio", "wav", "waveform", "mp3", "flac"]

assert set(PAPER_MAPPING) == set(PAPER_EMOTIONS)
assert set(MAJORITY_EMOTIONS).isdisjoint(set(MINORITY_EMOTIONS))


## 3. File inventory and schema

This section inspects Parquet metadata without loading the audio column into memory.


In [ ]:
def gib(n):
    return n / (1024 ** 3)

all_files = sorted([p for p in RAW_ROOT.rglob("*") if p.is_file()])
parquet_files = sorted(RAW_ROOT.rglob("*.parquet"))

print(f"Files: {len(all_files):,}")
print(f"Parquet files: {len(parquet_files):,}")
print(f"Total size: {gib(sum(p.stat().st_size for p in all_files)):.2f} GiB")

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found under {RAW_ROOT}")

file_preview = pd.DataFrame({
    "relative_path": [str(p.relative_to(RAW_ROOT)) for p in all_files[:80]],
    "suffix": [p.suffix for p in all_files[:80]],
    "size_mib": [round(p.stat().st_size / (1024 ** 2), 2) for p in all_files[:80]],
})
display(file_preview)


In [ ]:
parquet_summary = []
for path in tqdm(parquet_files, desc="Inspecting shards"):
    pf = pq.ParquetFile(path)
    parquet_summary.append({
        "source_parquet": str(path.relative_to(RAW_ROOT)),
        "num_rows": pf.metadata.num_rows,
        "num_row_groups": pf.metadata.num_row_groups,
        "size_mib": round(path.stat().st_size / (1024 ** 2), 2),
    })

parquet_summary = pd.DataFrame(parquet_summary)
parquet_summary.to_csv(OUTPUT_ROOT / "parquet_file_summary.csv", index=False)
print("Total rows from metadata:", f"{parquet_summary['num_rows'].sum():,}")
display(parquet_summary.head(20))


In [ ]:
first_file = parquet_files[0]
first_pf = pq.ParquetFile(first_file)
schema = first_pf.schema_arrow
schema_df = pd.DataFrame({
    "column": schema.names,
    "type": [str(field.type) for field in schema],
})

print("First shard:", first_file.relative_to(RAW_ROOT))
display(schema_df)


## 4. Load lightweight metadata

The manifest stores row locators and lightweight metadata only. It does not duplicate audio.


In [ ]:
def norm_name(s):
    return str(s).strip().lower().replace(" ", "_").replace("-", "_")

def first_existing(columns, candidates):
    lookup = {norm_name(c): c for c in columns}
    for cand in candidates:
        if norm_name(cand) in lookup:
            return lookup[norm_name(cand)]
    return None

def is_heavy_col(field):
    name = norm_name(field.name)
    audio_like = any(k in name for k in AUDIO_CANDIDATES + ["bytes"])
    binary_like = pa.types.is_binary(field.type) or pa.types.is_large_binary(field.type)
    return audio_like or binary_like

all_schema_cols = schema.names
metadata_cols = [field.name for field in schema if not is_heavy_col(field)]
skipped_cols = [field.name for field in schema if is_heavy_col(field)]

print("Metadata columns:", metadata_cols)
print("Skipped heavy/audio columns:", skipped_cols)


In [ ]:
def read_metadata_shard(path, columns):
    rel = str(path.relative_to(RAW_ROOT))
    pf = pq.ParquetFile(path)
    if columns:
        df = pq.read_table(path, columns=columns).to_pandas()
    else:
        df = pd.DataFrame(index=np.arange(pf.metadata.num_rows))
    df.insert(0, "row_in_file", np.arange(len(df), dtype=np.int64))
    df.insert(0, "source_parquet", rel)
    return df

parts = []
for path in tqdm(parquet_files, desc="Loading metadata only"):
    parts.append(read_metadata_shard(path, metadata_cols))

manifest = pd.concat(parts, ignore_index=True)
print("Rows:", f"{len(manifest):,}")
display(manifest.head())


## 5. Basic columns and stable IDs

`sample_id` should track the original utterance when possible. If the dataset has a `file` column, use it. Otherwise, create a row-based ID from the Parquet shard and row index.


In [ ]:
file_col = first_existing(manifest.columns, FILE_CANDIDATES)
text_col = first_existing(manifest.columns, TEXT_CANDIDATES)
major_emotion_col = first_existing(manifest.columns, ["major_emotion", "emotion", "label", "primary_emotion"])

if file_col:
    manifest["sample_id"] = manifest[file_col].astype(str).str.replace(r"\.[A-Za-z0-9]+$", "", regex=True)
else:
    manifest["sample_id"] = [f"{Path(p).stem}_{i:06d}" for p, i in zip(manifest["source_parquet"], manifest["row_in_file"])]

if text_col:
    text = manifest[text_col].fillna("").astype(str)
    manifest["text_num_chars"] = text.str.len()
    manifest["text_num_words"] = text.str.split().str.len()
else:
    manifest["text_num_chars"] = np.nan
    manifest["text_num_words"] = np.nan

print("file_col:", file_col)
print("text_col:", text_col)
print("major_emotion_col:", major_emotion_col)
display(manifest[["sample_id", "source_parquet", "row_in_file"] + ([file_col] if file_col else []) + ([text_col] if text_col else [])].head())


In [ ]:
def extract_episode_id(value):
    s = str(value)
    m = re.match(r"(MSP-PODCAST_\d+)_", s)
    if m:
        return m.group(1)
    m = re.match(r"([^_]+_[^_]+)_", s)
    if m:
        return m.group(1)
    return Path(s).stem

if file_col:
    manifest["episode_id"] = manifest[file_col].map(extract_episode_id)
else:
    manifest["episode_id"] = manifest["source_parquet"]

print("Unique episode/group IDs:", manifest["episode_id"].nunique())
display(manifest[["sample_id", "episode_id"]].head())


## 6. Check the native 16-emotion soft distribution

This is the part that can look strange at first. A row has a score for every native emotion column because the target is a vector. The important checks are:

- Are the columns present?
- Are they numeric?
- Does the row sum behave like a distribution?
- Is the largest value meaningful compared with the small floor values?


In [ ]:
missing_native = [c for c in NATIVE_EMOTION_COLS if c not in manifest.columns]
present_native = [c for c in NATIVE_EMOTION_COLS if c in manifest.columns]

print("Present native emotion columns:", present_native)
print("Missing native emotion columns:", missing_native)

if missing_native:
    raise ValueError(
        "The expected native emotion columns are missing. "
        "Inspect the schema above and update NATIVE_EMOTION_COLS."
    )

for col in present_native:
    if not pd.api.types.is_numeric_dtype(manifest[col]):
        raise TypeError(f"Emotion column is not numeric: {col}")

native_values = manifest[present_native].fillna(0).astype(float)
manifest["native_emotion_mass"] = native_values.sum(axis=1)
manifest["native_label_argmax"] = native_values.idxmax(axis=1)
manifest["native_label_argmax_score"] = native_values.max(axis=1)

if major_emotion_col:
    manifest["native_label"] = manifest[major_emotion_col].fillna(manifest["native_label_argmax"]).astype(str)
else:
    manifest["native_label"] = manifest["native_label_argmax"]

print("Native emotion mass summary:")
display(manifest["native_emotion_mass"].describe())

print("Native label counts:")
display(manifest["native_label"].value_counts(dropna=False).rename_axis("native_label").reset_index(name="count"))

preview_cols = ["sample_id", "native_label", "native_emotion_mass", "native_label_argmax", "native_label_argmax_score"] + present_native
with pd.option_context("display.float_format", "{:.6f}".format):
    display(manifest[preview_cols].head(10))


In [ ]:
fig = plt.figure(figsize=(7, 4))
manifest["native_emotion_mass"].hist(bins=50)
plt.title("Row sum of native 16-emotion scores")
plt.xlabel("sum across native emotion columns")
plt.ylabel("rows")
plt.tight_layout()
plt.show()

native_mean = native_values.mean().sort_values(ascending=False).rename_axis("emotion").reset_index(name="mean_score")
display(native_mean)

fig = plt.figure(figsize=(9, 4))
native_mean.plot.bar(x="emotion", y="mean_score", legend=False, rot=45)
plt.title("Mean native emotion score")
plt.xlabel("native emotion")
plt.ylabel("mean score")
plt.tight_layout()
plt.show()


## 7. Build paper-compatible 9-class primary soft targets

The paper's primary task is not the full native 16-way label space. It uses nine primary emotions. We aggregate native columns into a 9-class distribution and normalize it.

The saved target columns are named:

```text
target_primary9_angry, ..., target_primary9_other
```


In [ ]:
for paper_emo, native_cols in PAPER_MAPPING.items():
    missing = [c for c in native_cols if c not in manifest.columns]
    if missing:
        raise ValueError(f"Missing native columns for {paper_emo}: {missing}")
    manifest[f"target_primary9_{paper_emo}"] = manifest[native_cols].fillna(0).astype(float).sum(axis=1)

primary9_cols = [f"target_primary9_{emo}" for emo in PAPER_EMOTIONS]
primary9_raw = manifest[primary9_cols].fillna(0).astype(float)
primary9_mass = primary9_raw.sum(axis=1)
primary9 = primary9_raw.div(primary9_mass.replace(0, np.nan), axis=0).fillna(0)
manifest[primary9_cols] = primary9
manifest["primary9_mass_before_norm"] = primary9_mass
manifest["primary_label_9"] = primary9.idxmax(axis=1).str.replace("target_primary9_", "", regex=False)
manifest["primary_label_9_score"] = primary9.max(axis=1)
manifest["is_majority_primary"] = manifest["primary_label_9"].isin(MAJORITY_EMOTIONS)
manifest["is_minority_primary"] = manifest["primary_label_9"].isin(MINORITY_EMOTIONS)

# This approximates low-consensus/no-agreement behavior for analysis. It is not an official no-agreement label.
NO_AGREEMENT_THRESHOLD = 0.50
manifest["low_agreement_like"] = manifest["primary_label_9_score"] < NO_AGREEMENT_THRESHOLD

with pd.option_context("display.float_format", "{:.6f}".format):
    display(manifest[["sample_id", "native_label", "primary_label_9", "primary_label_9_score", "is_majority_primary", "low_agreement_like"] + primary9_cols].head(12))


In [ ]:
primary_counts = manifest["primary_label_9"].value_counts().reindex(PAPER_EMOTIONS, fill_value=0)
primary_summary = pd.DataFrame({
    "emotion": primary_counts.index,
    "count": primary_counts.values,
    "fraction": primary_counts.values / len(manifest),
    "paper_group": ["majority" if e in MAJORITY_EMOTIONS else "minority" for e in primary_counts.index],
})

display(primary_summary)
primary_summary.to_csv(OUTPUT_ROOT / "primary9_label_distribution.csv", index=False)

fig = plt.figure(figsize=(8, 4))
primary_summary.plot.bar(x="emotion", y="count", legend=False, rot=45)
plt.title("Paper-compatible primary 9-class hard labels from soft targets")
plt.xlabel("primary emotion")
plt.ylabel("rows")
plt.tight_layout()
plt.show()

print("Rows with low-agreement-like primary max score <", NO_AGREEMENT_THRESHOLD, ":", int(manifest["low_agreement_like"].sum()))


## 8. Secondary/multilabel targets and emotion attributes

The paper also tests additional supervision: secondary emotions and affective attributes. For this dataset:

- the native 16-emotion vector is kept as a secondary/multilabel soft target;
- `EmoAct`, `EmoVal`, and `EmoDom`, when present, are kept as regression targets for arousal/activation, valence, and dominance.

The saved secondary columns are named:

```text
target_secondary16_angry, ..., target_secondary16_happy
```


In [ ]:
secondary16_cols = []
secondary_raw = native_values.div(manifest["native_emotion_mass"].replace(0, np.nan), axis=0).fillna(0)
for col in present_native:
    out_col = f"target_secondary16_{col}"
    manifest[out_col] = secondary_raw[col]
    secondary16_cols.append(out_col)

present_attrs = [c for c in ATTRIBUTE_COLS if c in manifest.columns]
attr_target_cols = []
for col in present_attrs:
    out_col = {
        "EmoAct": "target_attr_activation",
        "EmoVal": "target_attr_valence",
        "EmoDom": "target_attr_dominance",
    }.get(col, f"target_attr_{norm_name(col)}")
    manifest[out_col] = pd.to_numeric(manifest[col], errors="coerce")
    attr_target_cols.append(out_col)

print("Secondary/multilabel target columns:", secondary16_cols)
print("Attribute target columns:", attr_target_cols)
if attr_target_cols:
    display(manifest[attr_target_cols].describe())


## 9. Quality checks for our use case

The downstream model needs: audio locator, transcript text, primary soft labels, optional secondary labels, and optional attribute labels.


In [ ]:
summary = {
    "num_rows": int(len(manifest)),
    "num_parquet_files": int(len(parquet_files)),
    "file_col": file_col,
    "text_col": text_col,
    "major_emotion_col": major_emotion_col,
    "audio_or_heavy_cols_skipped": skipped_cols,
    "native_emotion_cols": present_native,
    "primary9_cols": primary9_cols,
    "secondary16_cols": secondary16_cols,
    "attribute_target_cols": attr_target_cols,
    "majority_emotions": MAJORITY_EMOTIONS,
    "minority_emotions": MINORITY_EMOTIONS,
    "paper_mapping": PAPER_MAPPING,
    "low_agreement_like_threshold": NO_AGREEMENT_THRESHOLD,
}

if text_col:
    summary.update({
        "missing_text_rows": int(manifest[text_col].isna().sum()),
        "empty_text_rows": int((manifest[text_col].fillna("").astype(str).str.len() == 0).sum()),
        "duplicate_sample_ids": int(manifest["sample_id"].duplicated().sum()),
    })

print(json.dumps(summary, indent=2))
(OUTPUT_ROOT / "dataset_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")


In [ ]:
if text_col:
    display(manifest[["text_num_chars", "text_num_words"]].describe())
    fig = plt.figure(figsize=(7, 4))
    manifest["text_num_words"].dropna().clip(upper=120).hist(bins=50)
    plt.title("Transcript length distribution, clipped at 120 words")
    plt.xlabel("words")
    plt.ylabel("rows")
    plt.tight_layout()
    plt.show()

if "audio" in all_schema_cols:
    print("Audio is present in the Parquet schema and intentionally not loaded into the manifest.")


## 10. Split policy

The Hugging Face repository exposes one `train` split, so this notebook creates project train/dev/test manifests.

Default: group-based split by `episode_id`, which reduces leakage between adjacent utterances from the same podcast episode. If this gives unacceptable class imbalance, change `SPLIT_MODE` to `"stratified"` and rerun.


In [ ]:
SEED = 42
TRAIN_FRAC = 0.80
DEV_FRAC = 0.10
TEST_FRAC = 0.10
SPLIT_MODE = "group"  # "group", "stratified", or "random"

assert abs(TRAIN_FRAC + DEV_FRAC + TEST_FRAC - 1.0) < 1e-9


In [ ]:
def stratified_split(df, label_col):
    labels = df[label_col]
    counts = labels.value_counts()
    stratify = labels if len(counts) > 1 and counts.min() >= 2 else None
    train_idx, temp_idx = train_test_split(
        df.index,
        train_size=TRAIN_FRAC,
        random_state=SEED,
        shuffle=True,
        stratify=stratify,
    )
    temp = df.loc[temp_idx]
    temp_counts = temp[label_col].value_counts()
    temp_stratify = temp[label_col] if len(temp_counts) > 1 and temp_counts.min() >= 2 else None
    dev_frac_of_temp = DEV_FRAC / (DEV_FRAC + TEST_FRAC)
    dev_idx, test_idx = train_test_split(
        temp.index,
        train_size=dev_frac_of_temp,
        random_state=SEED,
        shuffle=True,
        stratify=temp_stratify,
    )
    return train_idx, dev_idx, test_idx


def random_split(df):
    train_idx, temp_idx = train_test_split(df.index, train_size=TRAIN_FRAC, random_state=SEED, shuffle=True)
    temp = df.loc[temp_idx]
    dev_frac_of_temp = DEV_FRAC / (DEV_FRAC + TEST_FRAC)
    dev_idx, test_idx = train_test_split(temp.index, train_size=dev_frac_of_temp, random_state=SEED, shuffle=True)
    return train_idx, dev_idx, test_idx


def grouped_split(df, group_col):
    groups = df[group_col].fillna("__missing_group__").astype(str)
    gss1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=SEED)
    train_pos, temp_pos = next(gss1.split(df, groups=groups))
    train_idx = df.index[train_pos]
    temp = df.iloc[temp_pos]
    temp_groups = temp[group_col].fillna("__missing_group__").astype(str)
    dev_frac_of_temp = DEV_FRAC / (DEV_FRAC + TEST_FRAC)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=dev_frac_of_temp, random_state=SEED)
    dev_pos, test_pos = next(gss2.split(temp, groups=temp_groups))
    return train_idx, temp.index[dev_pos], temp.index[test_pos]

if SPLIT_MODE == "group":
    train_idx, dev_idx, test_idx = grouped_split(manifest, "episode_id")
elif SPLIT_MODE == "stratified":
    train_idx, dev_idx, test_idx = stratified_split(manifest, "primary_label_9")
elif SPLIT_MODE == "random":
    train_idx, dev_idx, test_idx = random_split(manifest)
else:
    raise ValueError(f"Unknown SPLIT_MODE: {SPLIT_MODE}")

manifest["split"] = ""
manifest.loc[train_idx, "split"] = "train"
manifest.loc[dev_idx, "split"] = "dev"
manifest.loc[test_idx, "split"] = "test"

split_summary = manifest.groupby("split").size().rename("rows").reset_index()
split_summary["fraction"] = split_summary["rows"] / len(manifest)
display(split_summary)


In [ ]:
split_label_summary = (
    manifest.groupby(["split", "primary_label_9"])
    .size()
    .rename("rows")
    .reset_index()
)

split_label_pivot = split_label_summary.pivot(index="primary_label_9", columns="split", values="rows").fillna(0).astype(int)
split_label_pivot = split_label_pivot.reindex(PAPER_EMOTIONS)
display(split_label_pivot)

if SPLIT_MODE == "group":
    overlap = {}
    for a in ["train", "dev", "test"]:
        ga = set(manifest.loc[manifest["split"] == a, "episode_id"].astype(str))
        for b in ["train", "dev", "test"]:
            if a < b:
                gb = set(manifest.loc[manifest["split"] == b, "episode_id"].astype(str))
                overlap[f"{a}_{b}"] = len(ga & gb)
    print("Episode/group overlap:", overlap)

missing_by_split = {}
for split in ["train", "dev", "test"]:
    present = set(manifest.loc[manifest["split"] == split, "primary_label_9"])
    missing_by_split[split] = [e for e in PAPER_EMOTIONS if e not in present]
print("Missing primary classes by split:", missing_by_split)


## 11. Save manifests and target metadata

The split manifests become the interface for feature extraction and training. Feature extractors can read `source_parquet` + `row_in_file` to recover the original audio/text, then save frozen Whisper/RoBERTa hidden states under `data/processed/podcast`.


In [ ]:
manifest = manifest.sort_values(["split", "source_parquet", "row_in_file"]).reset_index(drop=True)

all_parquet_path = MANIFEST_ROOT / "podcast_manifest_all.parquet"
all_csv_path = MANIFEST_ROOT / "podcast_manifest_all.csv"
manifest.to_parquet(all_parquet_path, index=False)
manifest.to_csv(all_csv_path, index=False)

for split_name in ["train", "dev", "test"]:
    split_df = manifest[manifest["split"] == split_name].copy()
    split_df.to_parquet(MANIFEST_ROOT / "splits" / f"{split_name}.parquet", index=False)
    split_df.to_csv(MANIFEST_ROOT / "splits" / f"{split_name}.csv", index=False)

split_summary.to_csv(MANIFEST_ROOT / "split_summary.csv", index=False)
split_label_summary.to_csv(MANIFEST_ROOT / "split_primary9_summary.csv", index=False)
split_label_pivot.to_csv(MANIFEST_ROOT / "split_primary9_pivot.csv")

manifest_config = {
    "raw_root": str(RAW_ROOT.relative_to(PROJECT_ROOT)),
    "manifest_root": str(MANIFEST_ROOT.relative_to(PROJECT_ROOT)),
    "processed_root": str(PROCESSED_ROOT.relative_to(PROJECT_ROOT)),
    "file_col": file_col,
    "text_col": text_col,
    "audio_cols_skipped": skipped_cols,
    "row_locator_cols": ["source_parquet", "row_in_file"],
    "group_col": "episode_id",
    "split_mode": SPLIT_MODE,
    "target_spaces": {
        "primary_9": {
            "emotions": PAPER_EMOTIONS,
            "columns": primary9_cols,
            "hard_label_col": "primary_label_9",
            "majority_emotions": MAJORITY_EMOTIONS,
            "minority_emotions": MINORITY_EMOTIONS,
            "mapping_from_native_16": PAPER_MAPPING,
            "loss": "KL divergence or soft cross entropy",
        },
        "secondary_16_multilabel": {
            "native_emotions": present_native,
            "columns": secondary16_cols,
            "hard_label_col": "native_label",
            "loss": "BCE-with-logits for multilabel or KL if treated as a distribution",
        },
        "attributes": {
            "source_columns": present_attrs,
            "columns": attr_target_cols,
            "loss": "MSE or SmoothL1 regression",
        },
    },
    "low_agreement_like_col": "low_agreement_like",
    "low_agreement_like_threshold": NO_AGREEMENT_THRESHOLD,
}

(MANIFEST_ROOT / "manifest_config.json").write_text(json.dumps(manifest_config, indent=2), encoding="utf-8")
(MANIFEST_ROOT / "paper_emotion_mapping.json").write_text(json.dumps(PAPER_MAPPING, indent=2), encoding="utf-8")
(MANIFEST_ROOT / "native_emotion_columns.json").write_text(json.dumps(present_native, indent=2), encoding="utf-8")

print("Saved:")
for path in [
    all_parquet_path,
    all_csv_path,
    MANIFEST_ROOT / "splits" / "train.parquet",
    MANIFEST_ROOT / "splits" / "dev.parquet",
    MANIFEST_ROOT / "splits" / "test.parquet",
    MANIFEST_ROOT / "manifest_config.json",
]:
    print("-", path.relative_to(PROJECT_ROOT))


## 12. Final sanity checks


In [ ]:
checks = {}
for a in ["train", "dev", "test"]:
    ids_a = set(manifest.loc[manifest["split"] == a, "sample_id"])
    for b in ["train", "dev", "test"]:
        if a < b:
            ids_b = set(manifest.loc[manifest["split"] == b, "sample_id"])
            checks[f"{a}_{b}_sample_id_overlap"] = len(ids_a & ids_b)

checks.update({
    "rows_total": int(len(manifest)),
    "rows_with_split": int((manifest["split"] != "").sum()),
    "has_text": text_col is not None,
    "has_primary9_soft_targets": bool(primary9_cols),
    "has_secondary16_targets": bool(secondary16_cols),
    "has_attribute_targets": bool(attr_target_cols),
})

print(json.dumps(checks, indent=2))

if any(v for k, v in checks.items() if k.endswith("sample_id_overlap")):
    print("WARNING: sample_id overlap exists across splits.")

if text_col is None:
    print("WARNING: no transcript column detected. RoBERTa-side feature extraction will need transcripts from another source.")


## 13. Optional: inspect one audio payload

This reads one audio row only to confirm how feature extraction should access the audio later.


In [ ]:
audio_cols = [c for c in all_schema_cols if any(k in norm_name(c) for k in AUDIO_CANDIDATES)]
print("Audio-like columns:", audio_cols)

if audio_cols:
    audio_col = audio_cols[0]
    batch = next(pq.ParquetFile(first_file).iter_batches(batch_size=1, columns=[audio_col]))
    value = batch.to_pydict()[audio_col][0]
    print("Column:", audio_col)
    print("Python type:", type(value))
    if isinstance(value, dict):
        print("Keys:", list(value.keys()))
        for key, item in value.items():
            if isinstance(item, (bytes, bytearray)):
                print(f"{key}: {len(item):,} bytes")
            else:
                print(f"{key}: {repr(item)[:200]}")
    else:
        print(repr(value)[:500])
else:
    print("No audio-like column detected in the schema.")


## 14. How this feeds the model code

Training should use:

```text
data/manifests/podcast/splits/train.parquet
/data/manifests/podcast/splits/dev.parquet
/data/manifests/podcast/splits/test.parquet
```

Main task:

```text
target_primary9_* -> 9-class soft distribution, trained with KL divergence or soft cross entropy
```

Optional multitask heads:

```text
target_secondary16_* -> secondary/multilabel soft targets
target_attr_activation, target_attr_valence, target_attr_dominance -> regression targets
```

Majority/minority metadata:

```text
is_majority_primary, is_minority_primary
```

For the first frozen-feature pipeline, feature extraction should cache Whisper and RoBERTa hidden states under:

```text
data/processed/podcast/features/whisper_large_v3/...
data/processed/podcast/features/roberta_large/...
```
